# Disease mapping with neural processes

References: 
- 10.1371/journal.pcbi.1000724
- 10.7910/DVN/Z29FR0 (data)



In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from pyDataverse.api import DataAccessApi, NativeApi

token = os.environ.get("DATAVERSE_TOKEN")
base_url = "https://dataverse.harvard.edu/"
doi = "doi:10.7910/DVN/Z29FR0"
data_dir = Path("./data")


In [ ]:
# download dataset
api = NativeApi(base_url, token)
data_api = DataAccessApi(base_url, token)

dataset = api.get_dataset(doi)
dataset.json()

files = dataset.json()["data"]["latestVersion"]["files"]


data_dir.mkdir(parents=True, exist_ok=True)
for file in files:
    filename = file["dataFile"]["filename"]
    file_id = file["dataFile"]["persistentId"]
    response = data_api.get_datafile(file_id)
    print(filename)
    print("success" if response.status_code == 200 else "failed")
    with open(data_dir / filename, "wb") as f:
        f.write(response.content)


In [ ]:
dataset_file = data_dir / "00 Africa 1900-2015 SSA PR database (260617).tab"
df = pd.read_csv(dataset_file, sep="\t")

df.info()


In [ ]:
for column in df.columns:
    np.is_busday
    if pd.api.types.is_numeric_dtype(df[column]):
        print(f"{column}: min={df[column].min()} max={df[column].max()}")
    else:
        df[column] = df[column].str.strip().str.casefold()
        print(f"{column}:", sorted(df[column].unique()))


In [ ]:
def build_dataset(df: pd.DataFrame, year: int | None = None, month: int | None = None):
    df = df.query("AREATYPE=='point'")

    t = df.MM + (df.YY - df.YY.min()) * 12
    df["t"] = t

    if month is not None:
        assert year is not None, "If month is specified year cannot be None."
        df = df.query("YY==@year & MM==@month")
    elif year is not None:
        df = df.query("YY==@year")

    s = np.stack([df["Lat"], df["Long"], df["t"]], axis=-1)
    Nt = df["Pf"].to_numpy()
    Np = df["Ex"].to_numpy()

    return df, {"s": s, "Nt": Nt, "Np": Np}


dataset, numpy_dataset = build_dataset(df)

dataset

In [ ]:
import plotly.express as px

fig = px.scatter_geo(
    dataset, lat="Lat", lon="Long", hover_name="ID", scope="africa", color="t"
)
fig.show()